In [12]:
##### SI Table 3: performance of pre-calibration aggregation test

from pathlib import Path
import pandas as pd
import numpy as np
from glob import glob
from sklearn.metrics import r2_score

In [13]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent

# Import sub-national data
subnational_capital = pd.read_csv(f"{cd}/Results/Raster_models/intermediate_outputs/subnational_scaling_factors_capital.csv")
subnational_labor = pd.read_csv(f"{cd}/Results/Raster_models/intermediate_outputs/subnational_scaling_factors_labor.csv")

subnational_capital_null = pd.read_csv(f"{cd}/Results/Raster_models/country_avg_model/capital_subnational_agg.csv")
subnational_labor_null = pd.read_csv(f"{cd}/Results/Raster_models/country_avg_model/labor_subnational_agg.csv")

In [14]:
### Data prep

# clear missing data (subnational capital has a lot because proxy countries witheld)
subnational_capital = subnational_capital.dropna()
subnational_labor = subnational_labor.dropna()
subnational_capital_null = subnational_capital_null.dropna()
subnational_labor_null = subnational_labor_null.dropna()

# add actual to country average model 
subnational_capital_null = subnational_capital_null.merge(subnational_capital[['PROJ_ID', 'sub_national_total_capital_USD']], on='PROJ_ID', how='inner')
subnational_labor_null = subnational_labor_null.merge(subnational_labor[['PROJ_ID', 'sub_national_total_jobs']], on='PROJ_ID', how='inner')

In [15]:
##### Table

def pbias(actual, pred):
    return 100 * (pred - actual).sum() / actual.sum()

def coverage(actual, p10, p90):
    return 100 * ((actual >= p10) & (actual <= p90)).mean()

specs = [
    ("Capital", "Intensity model", subnational_capital,
     "sub_national_total_capital_USD", "capital_USD", "capital_USD_p10", "capital_USD_p90"),
    ("Capital", "Null model", subnational_capital_null,
     "sub_national_total_capital_USD", "capital_USD", None, None),
    ("Labor", "Intensity model", subnational_labor,
     "sub_national_total_jobs", "jobs", "jobs_p10", "jobs_p90"),
    ("Labor", "Null model", subnational_labor_null,
     "sub_national_total_jobs", "jobs", None, None),
]

rows = []
for model, spec, df, act, pred, p10, p90 in specs:
    cols = [c for c in (act, pred, p10, p90) if c is not None]
    d = np.log1p(df[cols].dropna())   # metrics computed on log1p scale
    rows.append({
        "Model": model,
        "Specification": spec,
        "n": len(d),
        "R2": r2_score(d[act], d[pred]),
        "PBIAS (%)": pbias(d[act], d[pred]),
        "QRF coverage (%)": coverage(d[act], d[p10], d[p90]) if p10 else np.nan,
    })

metrics = pd.DataFrame(rows).round(2)
metrics

,Model,Specification,n,R2,PBIAS (%),QRF coverage (%)
0,Capital,Intensity model,4951,0.65,0.72,87.40
1,Capital,Null model,4951,0.60,-1.22,NaN
2,Labor,Intensity model,12301,0.84,-3.96,78.46
3,Labor,Null model,12301,0.45,-10.48,NaN
